# PubMed Bioinformatics Abstracts Collection

Collect bioinformatics abstracts from PubMed (2020–2026) for LLM fine-tuning.

**Pipeline:**
1. Install dependencies
2. Configure NCBI Entrez & query parameters
3. Define helpers (parsing + retry logic)
4. Build time-sliced queries & fetch abstracts → JSONL
5. Merge slices → HuggingFace Dataset / Parquet
6. Preview & push to HuggingFace Hub

## Step 1 — Install Dependencies

In [1]:
%pip -q install biopython tqdm lxml datasets pandas pyarrow huggingface_hub

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.2 which is incompatible.
open-interpreter 0.1.3 requires huggingface-hub<0.17.0,>=0.16.4, but you have huggingface-hub 1.11.0 which is incompatible.
open-interpreter 0.1.3 requires openai<0.28.0,>=0.27.8, but you have openai 1.7.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Configure NCBI Entrez & Query Parameters

Set your NCBI email (required by their policy), API key (optional, gives higher rate limits), and define the search query and date range.

In [2]:
import os, time, json, re, random
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
from calendar import monthrange
from urllib.error import HTTPError, URLError

from Bio import Entrez
from tqdm.auto import tqdm

# REQUIRED by NCBI policies
Entrez.email = os.environ.get("NCBI_EMAIL", "your_email@example.com")  # <-- CHANGE THIS
# Optional but recommended (higher rate limits)
Entrez.api_key = os.environ.get("NCBI_API_KEY")

# High-precision base query
ENGLISH_ONLY = False  # set True if you want only english
BASE_QUERY = 'bioinformatics[MeSH Terms] AND hasabstract[text] AND journal article[pt]'
if ENGLISH_ONLY:
    BASE_QUERY += " AND english[la]"

DATE_FIELD = "dp"
START_YEAR = 2020
END_YEAR = 2026

# Fetch tuning
BATCH_SIZE_FETCH = 200
SLEEP_SECONDS = 0.4
MAX_RETRIES = 8

OUT_DIR = Path("out/pubmed_bioinformatics_abstracts")
SLICE_DIR = OUT_DIR / "slices_jsonl"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SLICE_DIR.mkdir(parents=True, exist_ok=True)

print("Entrez.email:", Entrez.email)
print("Entrez.api_key:", "set" if Entrez.api_key else "not set")
print("BASE_QUERY:", BASE_QUERY)
print("Date range:", START_YEAR, "-", END_YEAR)
print("Output dir:", OUT_DIR)

Entrez.email: your_email@example.com
Entrez.api_key: not set
BASE_QUERY: bioinformatics[MeSH Terms] AND hasabstract[text] AND journal article[pt]
Date range: 2020 - 2026
Output dir: out\pubmed_bioinformatics_abstracts


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3 — Define Helpers (Parsing + Retry Logic)

- `parse_pubmed_article()` extracts pmid, title, abstract, year, journal, doi
- Retry wrappers handle transient NCBI errors with exponential backoff

In [3]:
def _text(x) -> str:
    return "" if x is None else str(x)

def parse_pubmed_article(article: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """
    Return dict with: pmid, title, abstract, year, journal, doi
    """
    try:
        medline = article.get("MedlineCitation", {})
        pmid = _text(medline.get("PMID", "")).strip()
        if not pmid:
            return None

        art = medline.get("Article", {}) or {}
        title = _text(art.get("ArticleTitle", "")).strip()

        abstract = ""
        abs_obj = art.get("Abstract")
        if abs_obj and "AbstractText" in abs_obj:
            parts = abs_obj["AbstractText"]
            abstract = " ".join(_text(p).strip() for p in parts if _text(p).strip())
            abstract = re.sub(r"\s+", " ", abstract).strip()

        if not abstract:
            return None

        journal_obj = art.get("Journal") or {}
        journal = _text(journal_obj.get("Title", "")).strip()

        year = ""
        pubdate = (journal_obj.get("JournalIssue") or {}).get("PubDate") or {}
        year = _text(pubdate.get("Year", "")).strip()
        if not year:
            medline_date = _text(pubdate.get("MedlineDate", "")).strip()
            m = re.search(r"(19|20)\d{2}", medline_date)
            year = m.group(0) if m else ""

        doi = ""
        for eloc in art.get("ELocationID", []) or []:
            try:
                if getattr(eloc, "attributes", {}).get("EIdType") == "doi":
                    doi = _text(eloc).strip()
                    break
            except Exception:
                pass

        return {
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "year": year,
            "journal": journal,
            "doi": doi,
            "source": "pubmed",
        }
    except Exception:
        return None

In [7]:
import socket
import http.client

RETRYABLE_EXCEPTIONS = (
    HTTPError,
    URLError,
    RuntimeError,
    http.client.RemoteDisconnected,
    http.client.IncompleteRead,
    socket.timeout,
    ConnectionError,
    TimeoutError,
)

def _sleep_backoff(attempt: int):
    time.sleep(min(60, (2 ** (attempt - 1)) * 1.0) + random.uniform(0, 0.5))

def esearch_count(term: str) -> int:
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            h = Entrez.esearch(db="pubmed", term=term, retmax=0)
            rec = Entrez.read(h)
            h.close()
            return int(rec["Count"])
        except RETRYABLE_EXCEPTIONS as e:
            last_err = e
            print(f"[esearch_count retry] attempt={attempt}/{MAX_RETRIES} -> {type(e).__name__}: {e}")
            _sleep_backoff(attempt)
    raise last_err

def esearch_ids(term: str, retmax: int = 9999):
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            h = Entrez.esearch(db="pubmed", term=term, retmax=retmax)
            rec = Entrez.read(h)
            h.close()
            return list(rec["IdList"])
        except RETRYABLE_EXCEPTIONS as e:
            last_err = e
            print(f"[esearch_ids retry] attempt={attempt}/{MAX_RETRIES} -> {type(e).__name__}: {e}")
            _sleep_backoff(attempt)
    raise last_err

def efetch_and_read(id_list):
    """Fetch + parse in one retry loop (IncompleteRead happens during read)."""
    ids = ",".join(id_list)
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            h = Entrez.efetch(db="pubmed", id=ids, rettype="xml", retmode="xml")
            try:
                data = Entrez.read(h)
            finally:
                h.close()
            return data
        except RETRYABLE_EXCEPTIONS as e:
            last_err = e
            print(f"[efetch_and_read retry] ids={len(id_list)} attempt={attempt}/{MAX_RETRIES} -> {type(e).__name__}: {e}")
            _sleep_backoff(attempt)
    raise last_err

print("Retry helpers ready (with IncompleteRead handling).")

Retry helpers ready (with IncompleteRead handling).


## Step 4 — Build Time-Sliced Queries

Split the date range into yearly slices (auto-splits into months if a year has >9999 results, PubMed's esearch limit).

In [5]:
def slice_query_year(year: int) -> str:
    return f'{BASE_QUERY} AND ("{year}"[{DATE_FIELD}] : "{year}"[{DATE_FIELD}])'

def slice_query_month(year: int, month: int) -> str:
    last_day = monthrange(year, month)[1]
    return (
        f'{BASE_QUERY} AND ("{year}/{month:02d}/01"[{DATE_FIELD}] : '
        f'"{year}/{month:02d}/{last_day:02d}"[{DATE_FIELD}])'
    )

slices: List[Tuple[str, str, int]] = []

for y in range(START_YEAR, END_YEAR + 1):
    qy = slice_query_year(y)
    cy = esearch_count(qy)
    if cy == 0:
        continue

    if cy <= 9999:
        slices.append((f"{y}", qy, cy))
        print(f"Year {y}: {cy}")
    else:
        print(f"Year {y}: {cy} (too many; splitting into months)")
        for m in range(1, 13):
            qm = slice_query_month(y, m)
            cm = esearch_count(qm)
            if cm == 0:
                continue
            if cm > 9999:
                raise RuntimeError(
                    f"Month slice still > 9999: {y}-{m:02d} has {cm}. "
                    "Need daily-slicing for this month."
                )
            slices.append((f"{y}-{m:02d}", qm, cm))
            print(f"  Month {y}-{m:02d}: {cm}")

total_planned = sum(c for _, _, c in slices)
print(f"\nSlices: {len(slices)}")
print(f"Total planned articles: {total_planned:,}")

Year 2020: 20144 (too many; splitting into months)
  Month 2020-01: 4186
  Month 2020-02: 1973
  Month 2020-03: 2023
  Month 2020-04: 1985
  Month 2020-05: 1925
  Month 2020-06: 2083
  Month 2020-07: 2137
  Month 2020-08: 1976
  Month 2020-09: 2196
  Month 2020-10: 2220
  Month 2020-11: 2125
  Month 2020-12: 2039
Year 2021: 21727 (too many; splitting into months)
  Month 2021-01: 5067
  Month 2021-02: 2118
  Month 2021-03: 2167
  Month 2021-04: 1955
  Month 2021-05: 2246
  Month 2021-06: 2125
  Month 2021-07: 2180
  Month 2021-08: 1959
  Month 2021-09: 2186
  Month 2021-10: 2158
  Month 2021-11: 2296
  Month 2021-12: 2241
Year 2022: 17752 (too many; splitting into months)
  Month 2022-01: 4326
  Month 2022-02: 1824
  Month 2022-03: 1753
  Month 2022-04: 1601
  Month 2022-05: 1636
  Month 2022-06: 1801
  Month 2022-07: 1736
  Month 2022-08: 1756
  Month 2022-09: 1736
  Month 2022-10: 1669
  Month 2022-11: 1836
  Month 2022-12: 1884
Year 2023: 17499 (too many; splitting into months)
  Mo

## Step 5 — Fetch Abstracts (Resumable)

Downloads articles in batches per slice. Writes one JSONL file per slice. **Resumable** — re-running skips already-downloaded slices.

In [8]:
def slice_outfile(tag: str) -> Path:
    return SLICE_DIR / f"slice-{tag}.jsonl"

done_slices = 0
total_slices = len(slices)

for idx, (tag, q, c) in enumerate(slices, start=1):
    out_path = slice_outfile(tag)

    # Resume: skip if file exists and is non-empty
    if out_path.exists() and out_path.stat().st_size > 0:
        done_slices += 1
        continue

    pmids = esearch_ids(q, retmax=9999)
    if len(pmids) > 9999:
        raise RuntimeError(f"Unexpected PMID list > 9999 for slice {tag}: {len(pmids)}")

    with out_path.open("w", encoding="utf-8") as f:
        for i in tqdm(range(0, len(pmids), BATCH_SIZE_FETCH), desc=f"Slice {tag} ({idx}/{total_slices})"):
            batch = pmids[i:i+BATCH_SIZE_FETCH]
            data = efetch_and_read(batch)

            for article in data.get("PubmedArticle", []):
                rec = parse_pubmed_article(article)
                if rec is None:
                    continue
                rec["slice"] = tag
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")

            time.sleep(SLEEP_SECONDS)

    done_slices += 1
    print(f"Finished slice {tag} -> {out_path}")

print(f"\nDone: {done_slices}/{total_slices} slices")

Slice 2021-11 (23/73): 100%|██████████| 12/12 [00:57<00:00,  4.76s/it]


Finished slice 2021-11 -> out\pubmed_bioinformatics_abstracts\slices_jsonl\slice-2021-11.jsonl


Slice 2021-12 (24/73): 100%|██████████| 12/12 [01:17<00:00,  6.45s/it]


Finished slice 2021-12 -> out\pubmed_bioinformatics_abstracts\slices_jsonl\slice-2021-12.jsonl


Slice 2022-01 (25/73): 100%|██████████| 22/22 [02:15<00:00,  6.17s/it]


...

Finished slice 2025-11 -> out\pubmed_bioinformatics_abstracts\slices_jsonl\slice-2025-11.jsonl


Slice 2025-12 (72/73): 100%|██████████| 17/17 [02:51<00:00, 10.09s/it]


Finished slice 2025-12 -> out\pubmed_bioinformatics_abstracts\slices_jsonl\slice-2025-12.jsonl


Slice 2026 (73/73): 100%|██████████| 40/40 [05:10<00:00,  7.77s/it]

Finished slice 2026 -> out\pubmed_bioinformatics_abstracts\slices_jsonl\slice-2026.jsonl

Done: 73/73 slices


## Step 6 — Merge Slices → HuggingFace Dataset & Parquet

In [9]:
from datasets import load_dataset

slice_files = sorted(SLICE_DIR.glob("slice-*.jsonl"))
print(f"Slice files found: {len(slice_files)}")
assert len(slice_files) > 0, "No slice files found. Run Step 5 first."

ds = load_dataset("json", data_files=[str(p) for p in slice_files], split="train")
print(ds)
print("Columns:", ds.column_names)

PARQUET_DIR = OUT_DIR / "parquet"
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
parquet_path = PARQUET_DIR / "train.parquet"

ds.to_parquet(str(parquet_path))
print(f"Wrote: {parquet_path}")

Slice files found: 73


Generating train split: 174154 examples [00:01, 145821.75 examples/s]


Dataset({
    features: ['pmid', 'title', 'abstract', 'year', 'journal', 'doi', 'source', 'slice'],
    num_rows: 174154
})
Columns: ['pmid', 'title', 'abstract', 'year', 'journal', 'doi', 'source', 'slice']


Creating parquet from Arrow format: 100%|██████████| 4/4 [00:00<00:00,  4.06ba/s]

Wrote: out\pubmed_bioinformatics_abstracts\parquet\train.parquet


## Step 7 — Preview Sample Records

In [10]:
import random

n = min(5, len(ds))
idxs = random.sample(range(len(ds)), n)

for i in idxs:
    r = ds[i]
    print("\n---")
    print("pmid:", r.get("pmid"))
    print("year:", r.get("year"), "| journal:", (r.get("journal") or "")[:80])
    print("title:", (r.get("title") or "")[:200])
    abs_txt = (r.get("abstract") or "")
    print("abstract (first 500 chars):", abs_txt[:500])
    print("slice:", r.get("slice"))


---
pmid: 39504482
year: 2024 | journal: Briefings in bioinformatics
title: IMGT/RobustpMHC: robust training for class-I MHC peptide binding prediction.
abstract (first 500 chars): The accurate prediction of peptide-major histocompatibility complex (MHC) class I binding probabilities is a critical endeavor in immunoinformatics, with broad implications for vaccine development and immunotherapies. While recent deep neural network based approaches have showcased promise in peptide-MHC (pMHC) prediction, they have two shortcomings: (i) they rely on hand-crafted pseudo-sequence extraction, (ii) they do not generalize well to different datasets, which limits the practicality of 
slice: 2024-09

---
pmid: 32185441
year: 2020 | journal: Analytical and bioanalytical chemistry
title: Enantioanalysis of glutamine-a key factor in establishing the metabolomics process in gastric cancer.
abstract (first 500 chars): Gastric cancer is the second leading cause of death in the world. Early detection wi

## Step 8 — Authenticate & Push to HuggingFace Hub

Logs in with your HF token, creates the dataset repo + dataset card, and pushes. You'll be prompted to paste your token (hidden input).

In [14]:
import getpass
from huggingface_hub import HfApi, login, whoami

# --- Login (paste your HF token when prompted) ---
hf_token = getpass.getpass("Enter your HuggingFace token: ")
login(token=hf_token)

# --- Auto-detect username & build repo ID ---
user_info = whoami(token=hf_token)
hf_username = user_info["name"]
REPO_NAME = "pubmed-bioinformatics-abstracts"
HF_REPO_ID = f"{hf_username}/{REPO_NAME}"
print(f"Logged in as: {hf_username}")
print(f"Target repo: {HF_REPO_ID}")

# --- Create repo ---
api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
print(f"Repo ready: https://huggingface.co/datasets/{HF_REPO_ID}")

# --- Dataset card (README.md) ---
DATASET_CARD = f"""---
language:
  - en
license: cc-by-nc-sa-4.0
task_categories:
  - text-generation
  - text2text-generation
tags:
  - bioinformatics
  - pubmed
  - abstracts
  - life-sciences
  - nlp
  - llm-fine-tuning
size_categories:
  - 100K<n<1M
source_datasets:
  - pubmed
dataset_info:
  features:
    - name: pmid
      dtype: string
    - name: title
      dtype: string
    - name: abstract
      dtype: string
    - name: year
      dtype: string
    - name: journal
      dtype: string
    - name: doi
      dtype: string
    - name: source
      dtype: string
    - name: slice
      dtype: string
---

# PubMed Bioinformatics Abstracts ({START_YEAR}–{END_YEAR})

A curated collection of **{len(ds):,}** bioinformatics journal article abstracts from PubMed,
covering publications from **{START_YEAR}** to **{END_YEAR}**.

**Curated by:** Dr. Yash M Gupta

## Intended Use

Fine-tuning large language models (LLMs) on biomedical / bioinformatics domain text.

## Dataset Structure

| Column     | Type   | Description                          |
|------------|--------|--------------------------------------|
| `pmid`     | string | PubMed ID                            |
| `title`    | string | Article title                        |
| `abstract` | string | Full abstract text                   |
| `year`     | string | Publication year                     |
| `journal`  | string | Journal name                         |
| `doi`      | string | DOI (if available)                   |
| `source`   | string | Always "pubmed"                      |
| `slice`    | string | Time slice used during collection    |

## Collection Method

- **Source:** NCBI PubMed via Entrez E-utilities (Biopython)
- **Query:** `{BASE_QUERY}`
- **Date range:** {START_YEAR}–{END_YEAR} (date of publication)
- **Filters:** Only articles with abstracts; journal articles only

## License

This dataset is released under the
[Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/) license.

The abstracts themselves are sourced from PubMed/MEDLINE and are subject to the
[NLM Terms and Conditions](https://www.nlm.nih.gov/databases/download/terms_and_conditions.html).
Users must comply with NCBI/NLM usage policies when redistributing or using this data.

## Citation

If you use this dataset, please cite:

```bibtex
@dataset{{gupta_{START_YEAR}_pubmed_bioinformatics,
  author       = {{Gupta, Yash M}},
  title        = {{PubMed Bioinformatics Abstracts ({START_YEAR}--{END_YEAR})}},
  year         = {{2026}},
  publisher    = {{Hugging Face}},
  url          = {{https://huggingface.co/datasets/{HF_REPO_ID}}}
}}
```

Please also cite the original data source:

> Sayers EW, Bolton EE, Brister JR, et al. Database resources of the National Center for Biotechnology Information. *Nucleic Acids Research*. 2022;50(D1):D13–D25. doi:10.1093/nar/gkab1112
"""

api.upload_file(
    path_or_fileobj=DATASET_CARD.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
print("Dataset card uploaded.")

# --- Push parquet directly (avoids ds.push_to_hub bug with NoneType splits) ---
parquet_file = str(OUT_DIR / "parquet" / "train.parquet")
api.upload_file(
    path_or_fileobj=parquet_file,
    path_in_repo="data/train-00000-of-00001.parquet",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    commit_message=f"Upload {len(ds):,} bioinformatics abstracts ({START_YEAR}-{END_YEAR})",
)
print(f"Pushed {len(ds):,} records to: https://huggingface.co/datasets/{HF_REPO_ID}")

Logged in as: yashm
Target repo: yashm/pubmed-bioinformatics-abstracts
Repo ready: https://huggingface.co/datasets/yashm/pubmed-bioinformatics-abstracts


No files have been modified since last commit. Skipping to prevent empty commit.


Dataset card uploaded.


Processing Files (1 / 1): 100%|██████████|  167MB /  167MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.


Pushed 174,154 records to: https://huggingface.co/datasets/yashm/pubmed-bioinformatics-abstracts
